In [15]:
import pandas as pd
import re

# Read the Excel file
df = pd.read_excel('recruitment_agency/other_providers_leads.xlsx')

# 1. Create duplicate rows for any row having more than 1 email id
def split_emails(row):
    """Split rows with multiple emails separated by common delimiters"""
    email_col = 'Email'
    phone_col = 'Phone'  # Adjust if your column name is different
    
    if pd.isna(row[email_col]):
        return [row]
    
    # Split by common delimiters: comma, semicolon, space, pipe
    emails = re.split(r'[,;\s|]+', str(row[email_col]))
    emails = [e.strip() for e in emails if e.strip()]
    
    if len(emails) <= 1:
        return [row]
    
    # Create duplicate rows for each email
    rows = []
    for i, email in enumerate(emails):
        new_row = row.copy()
        new_row[email_col] = email
        # Keep phone number only for the first row, clear it for duplicates
        if i > 0:
            new_row[phone_col] = None  # or '' for empty string
        rows.append(new_row)
    return rows

# Apply the splitting function
expanded_rows = []
for _, row in df.iterrows():
    expanded_rows.extend(split_emails(row))

df = pd.DataFrame(expanded_rows).reset_index(drop=True)

# 2. Save business name's first part using - or | as delimiter
def extract_business_name(name):
    """Extract the first part of business name before - or |"""
    if pd.isna(name):
        return name
    name = str(name)
    # Split by - or | and take the first part
    parts = re.split(r'\s*[-|:–,]\s*|\s+I\s+', name)
    return parts[0].strip()

df['Name'] = df['Name'].apply(extract_business_name)

# 3. Filter out email records matching the sentry pattern
def is_sentry_email(email):
    """Check if email matches sentry pattern"""
    if pd.isna(email):
        return False
    email = str(email).lower()
    # Pattern: 32 hex characters @ sentry domains
    pattern = r'^[a-f0-9]{32}@sentry.*\.(wixpress\.com|io)$'
    return bool(re.match(pattern, email))

# Filter out sentry emails
df = df[~df['Email'].apply(is_sentry_email)].reset_index(drop=True)

# 4. Remove duplicate rows with same Name and Email
print(f"Rows before removing duplicates: {len(df)}")
df = df.drop_duplicates(subset=['Name', 'Email'], keep='first').reset_index(drop=True)
print(f"Rows after removing duplicates: {len(df)}")

# 5. Filter out Zoho users
def is_zoho_user(row):
    """Check if the lead uses Zoho MX records"""
    mx_col = 'MXRecords'  # Adjust column name if different
    
    if pd.isna(row[mx_col]):
        return False
    
    mx_records = str(row[mx_col]).lower()
    # Check if MX records contain zoho
    return 'zoho.com' in mx_records

zoho_count = df[df.apply(is_zoho_user, axis=1)].shape[0]
print(f"\nFiltering out {zoho_count} Zoho users...")
df = df[~df.apply(is_zoho_user, axis=1)].reset_index(drop=True)

# 6. Separate Hostinger leads
def is_hostinger_lead(row):
    """Check if the lead uses Hostinger MX records"""
    mx_col = 'MXRecords'  # Adjust column name if different
    
    if pd.isna(row[mx_col]):
        return False
    
    mx_records = str(row[mx_col]).lower()
    # Check if MX records contain hostinger
    return 'hostinger' in mx_records

# Split into Hostinger and non-Hostinger leads
hostinger_df = df[df.apply(is_hostinger_lead, axis=1)].reset_index(drop=True)
other_df = df[~df.apply(is_hostinger_lead, axis=1)].reset_index(drop=True)

# Display results
print(f"\n=== SUMMARY ===")
print(f"Total Hostinger leads: {len(hostinger_df)}")
print(f"Total other leads (excluding Zoho): {len(other_df)}")
print(f"Total leads (excluding Zoho): {len(df)}")

print("\n=== First few Hostinger leads ===")
print(hostinger_df.head(10))

print("\n=== First few other leads ===")
print(other_df.head(10))

# Save to separate Excel files
hostinger_df.to_excel('recruitment_agency/hostinger_cleaned_leads.xlsx', index=False)
other_df.to_excel('recruitment_agency/other_providers_cleaned_leads.xlsx', index=False)
# df.to_excel('all_cleaned_leads.xlsx', index=False)

print("\n=== Files saved ===")
print("✓ hostinger_leads.xlsx - Contains all Hostinger leads")
print("✓ other_leads.xlsx - Contains all non-Hostinger, non-Zoho leads")
print("✓ all_cleaned_leads.xlsx - Contains all cleaned leads (excluding Zoho)")

Rows before removing duplicates: 1180
Rows after removing duplicates: 996

Filtering out 24 Zoho users...

=== SUMMARY ===
Total Hostinger leads: 140
Total other leads (excluding Zoho): 832
Total leads (excluding Zoho): 972

=== First few Hostinger leads ===
                                                 URL            Industry  \
0  https://www.google.com/maps/place/TEACHCRUIT+P...  Recruitment Agency   
1  https://www.google.com/maps/place/Cincos+Place...  Recruitment Agency   
2  https://www.google.com/maps/place/Bangalore+Jo...  Recruitment Agency   
3  https://www.google.com/maps/place/HRdians+Cons...  Recruitment Agency   
4  https://www.google.com/maps/place/HRdians+Cons...  Recruitment Agency   
5  https://www.google.com/maps/place/ACTIVE+HR+PL...  Recruitment Agency   
6  https://www.google.com/maps/place/ACTIVE+HR+PL...  Recruitment Agency   
7  https://www.google.com/maps/place/Best+recruit...  Recruitment Agency   
8  https://www.google.com/maps/place/Ritu+Placeme...  Rec